<a href="https://colab.research.google.com/github/Akshatha7710/smart-tea-estate-management-system/blob/soil_quality-analysis_and_predictive_modeling/Soil_Quality_Analysis_and_Predictive_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# STEMS - Smart Tea Estate Management System
# Advanced Rainfall & Wet Days Prediction Pipeline
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import TimeSeriesSplit, RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, r2_score

# ============================================================
# 1️⃣ PREPARE TIME INDEX
# ============================================================

clean_df["Date"] = pd.to_datetime(clean_df[["Year", "Month"]].assign(DAY=1))
clean_df = clean_df.sort_values("Date")
clean_df = clean_df.set_index("Date")

# ============================================================
# 2️⃣ FEATURE ENGINEERING
# ============================================================

# --- Lag Features ---
for lag in [1, 3, 12]:
    clean_df[f"Rain_lag{lag}"] = clean_df["Rainfall"].shift(lag)
    clean_df[f"Wet_lag{lag}"] = clean_df["WetDays"].shift(lag)

# --- Rolling Means ---
for window in [3, 6, 12]:
    clean_df[f"Rain_roll{window}"] = clean_df["Rainfall"].rolling(window).mean()
    clean_df[f"Wet_roll{window}"] = clean_df["WetDays"].rolling(window).mean()

# --- Cyclical Encoding for Month ---
clean_df["Month_sin"] = np.sin(2 * np.pi * clean_df.index.month / 12)
clean_df["Month_cos"] = np.cos(2 * np.pi * clean_df.index.month / 12)

# Drop NA from lagging
clean_df.dropna(inplace=True)

# ============================================================
# 3️⃣ FEATURES & TARGETS
# ============================================================

feature_cols = [col for col in clean_df.columns
                if col not in ["Year", "Month", "Rainfall", "WetDays"]]

X = clean_df[feature_cols]
y = clean_df[["Rainfall", "WetDays"]]

# ============================================================
# 4️⃣ TIME SERIES CROSS VALIDATION
# ============================================================

tscv = TimeSeriesSplit(n_splits=5)

# ============================================================
# 5️⃣ HYPERPARAMETER TUNING
# ============================================================

param_dist = {
    "estimator__n_estimators": [300, 400, 500, 600],
    "estimator__max_depth": [10, 15, 20, None],
    "estimator__min_samples_split": [2, 5, 10],
    "estimator__min_samples_leaf": [1, 2, 4]
}

base_rf = RandomForestRegressor(random_state=42)

multi_model = MultiOutputRegressor(base_rf)

search = RandomizedSearchCV(
    multi_model,
    param_distributions=param_dist,
    n_iter=20,
    cv=tscv,
    scoring="r2",
    random_state=42,
    n_jobs=-1
)

search.fit(X, y)

print("Best Parameters Found:")
print(search.best_params_)

best_model = search.best_estimator_

# ============================================================
# 6️⃣ CROSS-VALIDATION PERFORMANCE
# ============================================================

r2_scores = []
mae_scores = []

for train_index, test_index in tscv.split(X):

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    best_model.fit(X_train, y_train)
    preds = best_model.predict(X_test)

    r2_scores.append(r2_score(y_test, preds))
    mae_scores.append(mean_absolute_error(y_test, preds))

print("\nModel Performance:")
print("Average R2:", np.mean(r2_scores))
print("Average MAE:", np.mean(mae_scores))

# ============================================================
# 7️⃣ FINAL TRAINING ON FULL DATA
# ============================================================

best_model.fit(X, y)

# ============================================================
# 8️⃣ FEATURE IMPORTANCE (Rainfall Model Only)
# ============================================================

rain_importance = pd.Series(
    best_model.estimators_[0].feature_importances_,
    index=feature_cols
).sort_values()

plt.figure()
rain_importance.plot(kind="barh")
plt.title("Rainfall Feature Importance")
plt.show()

# ============================================================
# 9️⃣ FUTURE PREDICTION EXAMPLE
# ============================================================

future_prediction = best_model.predict(X.tail(1))
print("\nNext Month Prediction (Rainfall, WetDays):")
print(future_prediction)

NameError: name 'clean_df' is not defined